In [ ]:
CREATE OR REPLACE PROCEDURE PROC_COPY_BRONZE(
    P_STAGE STRING,
    P_FILE_FORMAT STRING,
    P_TABLES ARRAY  
    -- Example: ARRAY_CONSTRUCT(OBJECT_CONSTRUCT('table_name','BRONZE_CUSTOMERS_DELTA','subfolder','customers'))
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
AS
$$
from datetime import datetime, timedelta
from snowflake.snowpark import Session

def main(session: Session, P_STAGE: str, P_FILE_FORMAT: str, P_TABLES: list):

    results = []
    success = []
    failed = []

    # Step 1: Folder path (UTC)
    folder_path = (datetime.utcnow() - timedelta(days=0)).strftime("%Y/%b/%d/").title()
    session.sql(f"SELECT '📁 Using folder path: {folder_path}'").collect()

    # Step 2: Loop through each object
    for tbl in P_TABLES:
        table_name = tbl.get("table_name")
        subfolder = tbl.get("subfolder")

        if not table_name or not subfolder:
            failed.append(f"(missing params for entry: {tbl})")
            continue

        # Capitalize first letter for pattern consistency
        pattern_prefix = subfolder.capitalize()

        # Step 3: Build COPY INTO statement
        session.sql(f"TRUNCATE TABLE {table_name}").collect()
        copy_query = f"""
        COPY INTO {table_name}
        FROM @{P_STAGE}/{subfolder}/Delta_load/csv_files/{folder_path}
        FILE_FORMAT = (FORMAT_NAME = {P_FILE_FORMAT})
        MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
        PATTERN = '.*{pattern_prefix}_.*?/part-.*\\\\.parquet';
        """

        try:
            session.sql(copy_query).collect()
            success.append(table_name)
        except Exception as e:
            failed.append(f"{table_name}: {str(e)}")

    # Step 4: Commit or rollback based on outcome
    if len(failed) == 0:
        session.sql("COMMIT").collect()
        summary = f"✅ Loaded {len(success)} tables successfully: {', '.join(success)}"
    else:
        session.sql("ROLLBACK").collect()
        summary = f"⚠️ Partial load. Success: {len(success)}, Failed: {len(failed)} | Errors: {failed}"

    return f"{summary} | Folder path: {folder_path}"
$$;


In [ ]:
CALL PROC_COPY_BRONZE(
    'BRONZE_CSV_DELTA',
    'MY_PARQUET_FORMAT',
    ARRAY_CONSTRUCT(
        OBJECT_CONSTRUCT('table_name','BRONZE_CUSTOMERS_DELTA','subfolder','customers'),
        OBJECT_CONSTRUCT('table_name','BRONZE_PRODUCTS_DELTA','subfolder','products'),
        OBJECT_CONSTRUCT('table_name','BRONZE_ORDERS_DELTA','subfolder','orders')
    )
);


In [ ]:
select * from BRONZE_customers_DELTA

In [ ]:
CREATE OR REPLACE PROCEDURE PROC_DELTA_SILVER_CDC(
    DB STRING,
    SCHEMA STRING,
    TABLE_MAPPINGS ARRAY
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
AS
$$
from snowflake.snowpark import Session

def main(session: Session, DB, SCHEMA, TABLE_MAPPINGS):

    MAX_DATE = "2999-12-31 00:00:00"
    results = []

    for mapping in TABLE_MAPPINGS:

        bronze = f"{DB}.{SCHEMA}.{mapping['bronze_table']}"
        silver = f"{DB}.{SCHEMA}.{mapping['silver_table']}"
        key = mapping['key_column']

        # ---------------------------------------
        # 1. GET COMMON COLUMNS
        # ---------------------------------------
        bronze_cols = session.table(bronze).columns
        silver_cols = session.table(silver).columns

        common_cols = [c for c in bronze_cols if c in silver_cols]

        compare_cols = [
            c for c in common_cols
            if c.upper() not in [key.upper(), "START_DATE", "END_DATE", "LOAD_DATE", "DELETE_FLAG"]
        ]

        insert_cols = [
            c for c in common_cols
            if c.upper() != "LOAD_DATE"
        ]

        insert_col_list = ", ".join([f'"{c}"' for c in insert_cols])
        select_col_list = ", ".join([f'cs."{c}"' for c in insert_cols])

        # ---------------------------------------
        # 2. CHANGE CONDITION
        # ---------------------------------------
        if compare_cols:
            change_condition = " OR ".join([
                f"COALESCE(TO_VARCHAR(s.\"{c}\"),'') <> COALESCE(TO_VARCHAR(b.\"{c}\"),'')"
                for c in compare_cols
            ])
        else:
            change_condition = "FALSE"

        # ---------------------------------------
        # 3. CREATE CHANGE SET (TEMP TABLE)
        # ---------------------------------------
        change_set = f"{silver}_CHANGE_SET"

        session.sql(f"""
            CREATE OR REPLACE TRANSIENT TABLE {change_set} AS
            SELECT
                b.*,
                CASE
                    WHEN b.DELETE_FLAG = 'Y' AND s."{key}" IS NOT NULL THEN 'DELETE'
                    WHEN s."{key}" IS NULL AND b.DELETE_FLAG = 'N' THEN 'NEW'
                    WHEN b.DELETE_FLAG = 'N' AND s."{key}" IS NOT NULL AND ({change_condition}) THEN 'UPDATE'
                    ELSE 'UNCHANGED'
                END AS CHANGE_TYPE
            FROM {bronze} b
            LEFT JOIN {silver} s
              ON b."{key}" = s."{key}"
             AND s.END_DATE = '{MAX_DATE}'
        """).collect()

        # ---------------------------------------
        # 4. CLOSE RECORDS (DELETE + UPDATE)
        # ---------------------------------------
        session.sql(f"""
            MERGE INTO {silver} s
            USING {change_set} cs
            ON s."{key}" = cs."{key}"
               AND s.END_DATE = '{MAX_DATE}'
            WHEN MATCHED AND cs.CHANGE_TYPE IN ('DELETE','UPDATE')
            THEN UPDATE SET
                s.END_DATE = cs.START_DATE,
                s.LOAD_DATE = CURRENT_TIMESTAMP(),
                s.DELETE_FLAG = 'Y'
        """).collect()

        # ---------------------------------------
        # 5. INSERT NEW + UPDATED RECORDS
        # ---------------------------------------
        session.sql(f"""
            INSERT INTO {silver} ({insert_col_list}, LOAD_DATE)
            SELECT
                {select_col_list},
                CURRENT_TIMESTAMP()
            FROM {change_set} cs
            WHERE cs.CHANGE_TYPE IN ('NEW','UPDATE')
        """).collect()

        # ---------------------------------------
        # 6. CLEANUP
        # ---------------------------------------
        session.sql(f"DROP TABLE IF EXISTS {change_set}").collect()

        results.append(f"{mapping['silver_table']} processed")

    return "CDC Completed Successfully (Change-Set Approach): " + ", ".join(results)

$$;

In [ ]:
CALL PROC_DELTA_SILVER_CDC(
    'DEV',
    'DATASCIENCE',
    ARRAY_CONSTRUCT(
        OBJECT_CONSTRUCT(
            'bronze_table','BRONZE_CUSTOMERS_DELTA',
            'silver_table','SILVER_CUSTOMERS_DELTA',
            'key_column','CUSTOMER_ID'
        ),
        OBJECT_CONSTRUCT(
            'bronze_table','BRONZE_PRODUCTS_DELTA',
            'silver_table','SILVER_PRODUCTS_DELTA',
            'key_column','PRODUCT_ID'
        ),
        OBJECT_CONSTRUCT(
            'bronze_table','BRONZE_ORDERS_DELTA',
            'silver_table','SILVER_ORDERS_DELTA',
            'key_column','ORDER_ID'
        )
    )
);

In [ ]:
select * from SILVER_customers_DELTA order by customer_id